In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv('BVBRC_genome_amr_simplified.csv')
df.shape

(83096, 4)

In [5]:
print(df['target'].value_counts())

target
0    57912
1    25184
Name: count, dtype: int64


In [6]:
df = df.drop(columns=['Genome Name'])

In [9]:
X = df[['Antibiotic', 'Strain Name']].copy()
y = df['target']

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 80-20 split
    random_state=42,      # reproducible
    stratify=y            # keep class ratio same
)

In [11]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# Fit on training set only
X_train_encoded = encoder.fit_transform(X_train)
# Apply to test set
X_test_encoded = encoder.transform(X_test)

In [12]:
import numpy as np

neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

print("Scale positive weight:", scale_pos_weight)

Scale positive weight: 2.299548319849109


In [13]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight='balanced',  # automatically handles imbalance
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_encoded, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [38]:
import pickle
with open("random_forest_model.pkl", "wb") as f:
    pickle.dump(rf, f)

print("Model saved as random_forest_model.pkl")

Model saved as random_forest_model.pkl


In [14]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,  # handle imbalance
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train_encoded, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [15]:
from sklearn.metrics import classification_report, roc_auc_score

# Random Forest
y_pred_rf = rf.predict(X_test_encoded)
rf_probs = rf.predict_proba(X_test_encoded)[:, 1]

print("Random Forest Report:\n", classification_report(y_test, y_pred_rf))
print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_probs))

# XGBoost
y_pred_xgb = xgb.predict(X_test_encoded)
xgb_probs = xgb.predict_proba(X_test_encoded)[:, 1]

print("XGBoost Report:\n", classification_report(y_test, y_pred_xgb))
print("XGBoost ROC-AUC:", roc_auc_score(y_test, xgb_probs))

Random Forest Report:
               precision    recall  f1-score   support

           0       0.85      0.74      0.79     11583
           1       0.54      0.71      0.61      5037

    accuracy                           0.73     16620
   macro avg       0.70      0.72      0.70     16620
weighted avg       0.76      0.73      0.74     16620

Random Forest ROC-AUC: 0.7931321001931815
XGBoost Report:
               precision    recall  f1-score   support

           0       0.86      0.69      0.77     11583
           1       0.51      0.75      0.61      5037

    accuracy                           0.71     16620
   macro avg       0.69      0.72      0.69     16620
weighted avg       0.76      0.71      0.72     16620

XGBoost ROC-AUC: 0.7897936089650734


In [17]:
from sklearn.metrics import f1_score

f1 = f1_score(y_test, y_pred_rf)
print("F1-score:", f1)
f1 = f1_score(y_test, y_pred_xgb)
print("F1-score:", f1)

F1-score: 0.6108010646518417
F1-score: 0.6072351421188631


In [18]:
X = df[['Antibiotic', 'Strain Name']].copy()
y = df['target']

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# Ensure X_train and X_test are only categorical columns
# e.g., X_train = df[['Antibiotic', 'Species_Mapped']]

# Use sparse=True for memory efficiency (especially if many categories)
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on train and transform
X_train_ohe = encoder.fit_transform(X_train)
X_test_ohe = encoder.transform(X_test)

# Convert back to DataFrame with proper column names
ohe_columns = encoder.get_feature_names_out(X_train.columns)
X_train_ohe = pd.DataFrame(X_train_ohe, columns=ohe_columns, index=X_train.index)
X_test_ohe = pd.DataFrame(X_test_ohe, columns=ohe_columns, index=X_test.index)

print("X_train_ohe shape:", X_train_ohe.shape)
print("X_test_ohe shape:", X_test_ohe.shape)

X_train_ohe shape: (66476, 2581)
X_test_ohe shape: (16620, 2581)


In [21]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train_ohe, y_train)

import pandas as pd
print(pd.Series(y_train_res).value_counts())

target
0    46329
1    46329
Name: count, dtype: int64


In [22]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_res, y_train_res)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [24]:
X_train_res.columns = [str(c).replace('[','_').replace(']','_').replace('<','_') for c in X_train_res.columns]

In [25]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train_res, y_train_res)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [27]:
# Ensure all columns exist
for c in X_train_res.columns:
    if c not in X_test_ohe.columns:
        X_test_ohe[c] = 0  # add missing columns with 0

# Reorder columns to match training set
X_test_ohe = X_test_ohe[X_train_res.columns]

In [28]:
# Replace invalid characters in both train and test sets
def sanitize_cols(df):
    df = df.copy()
    df.columns = [str(c).replace('[','_').replace(']','_').replace('<','_') for c in df.columns]
    return df

X_train_res = sanitize_cols(X_train_res)
X_test_ohe = sanitize_cols(X_test_ohe)

In [30]:
y_pred_xgb = xgb.predict(X_test_ohe)
xgb_probs = xgb.predict_proba(X_test_ohe)[:,1]

In [36]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder